In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_DIR))
print(PROJECT_DIR)

In [ ]:
from src.core.config import settings, update_path_settings

update_path_settings(PROJECT_DIR)

In [ ]:
from src.domain.pipelines.annotations import (
    clean_annotation_dataframe,
    load_annotation_file,
)

In [ ]:
def get_annotation_file(annotations_dir: Path, record_file: Path) -> Path | None:
    annotation_file = record_file.stem + ".txt"
    return_path = annotations_dir / annotation_file
    return return_path if return_path.exists() else None

In [ ]:
DATA_RAW_DIR = settings.DATA_RAW_DIR or Path("data/raw")
directories = [d for d in os.listdir(DATA_RAW_DIR) if (DATA_RAW_DIR / d).is_dir()]

for dir in directories:
    records_files = [
        DATA_RAW_DIR / dir / wav_file
        for wav_file in os.listdir(DATA_RAW_DIR / dir)
        if wav_file.endswith(".wav")
    ]
    for record_file in records_files:
        annotation_file = get_annotation_file(DATA_RAW_DIR / dir, record_file)
        if annotation_file is None:
            print(f"No annotation file found for {record_file}")
            continue

        try:
            raw_annotations = load_annotation_file(annotation_file)
            cleaned_annotations = clean_annotation_dataframe(raw_annotations)
            preprocessed_dir = settings.DATA_PREPROCESSED_DIR or Path(".") / dir
            (preprocessed_dir / dir).mkdir(parents=True, exist_ok=True)
            cleaned_annotations.to_csv(
                preprocessed_dir / dir / (record_file.stem + ".csv"), index=False
            )
        except Exception as e:
            print(f"Error processing {annotation_file}: {e}")